# สำรวจข้อมูล — Student Social Media & Mental Health Impact

โน้ตบุ๊กนี้ทำงานได้ด้วยตัวเอง ไม่ต้องพึ่งไฟล์ใน `src/` — โหลดข้อมูลจาก Kaggle เอง วาดกราฟเอง

เปิดจากโฟลเดอร์ `coding/`:

```fish
uv run jupyter lab
```

## 0. ตั้งค่า

In [1]:
%matplotlib inline

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")

# ฟอนต์ไทย — ใช้ตัวเดียวกับเล่ม (main.tex ใช้ TH Sarabun New) ไม่งั้นภาษาไทยจะขึ้นเป็นสี่เหลี่ยม
plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["TH Sarabun New", "Sarabun", "IBM Plex Sans Thai", "DejaVu Sans"],
    "font.size": 15,
    "axes.titlesize": "large",
    "axes.titleweight": "bold",
    "axes.titlelocation": "left",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "figure.dpi": 110,
    "savefig.dpi": 200,
    "savefig.bbox": "tight",
})

# ชุดสีที่ตรวจแล้วว่าคนตาบอดสีก็ยังแยกออก ใช้ไล่ตามลำดับ อย่าสลับ
BLUE, ORANGE, AQUA = "#2a78d6", "#eb6834", "#1baf7a"
plt.rcParams["axes.prop_cycle"] = plt.cycler(color=[BLUE, ORANGE, AQUA])

## 1. ดึงข้อมูลจาก Kaggle

ใช้ `kagglehub` (ลงไว้แล้วด้วย `uv add kagglehub`) ชุดข้อมูลนี้เป็น public
เลยโหลดได้เลยโดยไม่ต้องมี API token

`dataset_download()` จะโหลดไปเก็บไว้ที่ `~/.cache/kagglehub/` แล้วคืน path มาให้
รันซ้ำครั้งต่อไปจะใช้ของใน cache ไม่โหลดใหม่

> ถ้าวันหลังเจอชุดข้อมูลที่เป็น private หรือต้องกด accept rules ก่อน จะต้องมี token:
> kaggle.com → รูปโปรไฟล์ → Settings → API → Create New Token
> แล้วเอาไฟล์ `kaggle.json` ไปไว้ที่ `~/.kaggle/kaggle.json` (`chmod 600`)

In [2]:
import shutil
from pathlib import Path

import kagglehub

DATA_DIR = Path.cwd().parent / "data"   # coding/data/
DATA_DIR.mkdir(exist_ok=True)

cached = Path(kagglehub.dataset_download("shivasingh4945/student-social-media-and-mental-health-impact"))

# ก็อปออกจาก cache มาไว้ในโปรเจกต์ จะได้ไม่ผูกกับเครื่องนี้เครื่องเดียว
for src in sorted(cached.rglob("*.csv")):
    shutil.copy2(src, DATA_DIR / src.name)

csv_files = sorted(DATA_DIR.glob("*.csv"))
csv_files

/home/alexander_user/allmycoding/data-sci-year2/Project/proposal/coding/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[PosixPath('/home/alexander_user/allmycoding/data-sci-year2/Project/proposal/coding/data/Student Social Media And Mental Health Impact.csv')]

In [3]:
df = pd.read_csv(csv_files[0])
print(df.shape)   # (แถว, คอลัมน์)
df.head()

(5000, 13)


,Age,Gender,Country,Academic_Level,Most_Used_Platform,Purpose_Of_Use,Avg_Daily_Usage_Hours,Daily_Unlocks,Study_Hours,Physical_Activity_Hours,Sleep_Hours_Per_Night,Stress_Level,Mental_Health_Score
0,21,Male,Other,Undergraduate,Facebook,Networking,4.0,134,4.5,2.2,6.7,Medium,6.8
1,23,Female,Other,Graduate,LinkedIn,Education,1.6,73,7.0,2.4,8.6,Low,7.6
2,22,Male,Canada,Undergraduate,Instagram,Entertainment,4.6,166,4.0,1.8,6.7,Medium,7.0
3,18,Male,Other,High School,Snapchat,Entertainment,7.0,220,1.0,1.7,5.4,Very High,5.3
4,24,Female,Other,Graduate,Facebook,Networking,7.5,237,1.0,1.1,5.0,Very High,4.4


## 2. ในไฟล์มีอะไรบ้าง

| คอลัมน์ | ความหมาย |
|---|---|
| `Age` | อายุ |
| `Gender` | เพศ |
| `Country` | ประเทศ |
| `Academic_Level` | ระดับการศึกษา (High School / Undergraduate / Graduate) |
| `Most_Used_Platform` | แพลตฟอร์มที่ใช้มากที่สุด |
| `Purpose_Of_Use` | ใช้เพื่ออะไร |
| `Avg_Daily_Usage_Hours` | ชั่วโมงใช้โซเชียลเฉลี่ยต่อวัน |
| `Daily_Unlocks` | ปลดล็อกมือถือกี่ครั้งต่อวัน |
| `Study_Hours` | ชั่วโมงอ่านหนังสือ |
| `Physical_Activity_Hours` | ชั่วโมงกิจกรรมทางกาย |
| `Sleep_Hours_Per_Night` | ชั่วโมงนอนต่อคืน |
| `Stress_Level` | ระดับความเครียด — Low / Medium / High / Very High |
| `Mental_Health_Score` | คะแนนสุขภาพจิต **ยิ่งสูง = ยิ่งดี** (ระวังตีความกลับด้าน) |

In [ ]:
df.info()
print("\nค่าว่างต่อคอลัมน์:", df.isna().sum().sum(), "ช่อง")

df.describe().round(2)

In [ ]:
# คอลัมน์ที่เป็นข้อความ มีค่าอะไรบ้าง อย่างละกี่คน
for col in df.select_dtypes(exclude="number").columns:
    print(f"--- {col} ({df[col].nunique()} ค่า)")
    print(df[col].value_counts().head(6).to_string(), "\n")

## 3. กราฟใบแรก

ลองแก้ `bins` เป็น 5 / 20 / 60 แล้วรันซ้ำ จะเห็นว่าจำนวน bin เปลี่ยน "เรื่อง" ที่กราฟเล่าได้เลย

In [ ]:
fig, ax = plt.subplots(figsize=(7.2, 4.2))

sns.histplot(df, x="Avg_Daily_Usage_Hours", bins=20, color=BLUE, edgecolor="white", ax=ax)

median = df["Avg_Daily_Usage_Hours"].median()
ax.axvline(median, color="black", ls="--", lw=1.5)
ax.annotate(f"มัธยฐาน {median:.1f} ชม./วัน", (median, ax.get_ylim()[1] * 0.9),
            xytext=(6, 0), textcoords="offset points", fontweight="bold")

ax.set_title("นักศึกษาใช้โซเชียลมีเดียกี่ชั่วโมงต่อวัน")
ax.set_xlabel("ชั่วโมงต่อวัน")
ax.set_ylabel("จำนวนคน")
ax.xaxis.set_major_locator(mticker.MultipleLocator(2))
plt.show()

## 4. ตัวแปรไหนสัมพันธ์กับตัวไหน

heatmap สหสัมพันธ์ ใช้ดูภาพรวมก่อนว่าควรไปเจาะคู่ไหนต่อ
สีน้ำเงิน = สัมพันธ์ทางลบ (ตัวหนึ่งมาก อีกตัวน้อย), สีแดง = สัมพันธ์ทางบวก

**อ่านผลแล้วให้เอะใจด้วย** ตัวเลขในชุดข้อมูลนี้สูงผิดปกติสำหรับข้อมูลแบบสอบถามจริง
(เช่น ชั่วโมงใช้งาน ↔ จำนวนครั้งปลดล็อก r ≈ 0.96, ↔ ชั่วโมงอ่านหนังสือ r ≈ −0.88)
ข้อมูลสำรวจจากคนจริงแทบไม่เคยสวยขนาดนี้ — มีความเป็นไปได้สูงว่าชุดนี้ถูก **generate ขึ้นมา**
ไม่ใช่เก็บจากภาคสนาม ตรงนี้ต้องเขียนกำกับไว้ในเล่มด้วย ไม่งั้นจะสรุปเกินจริง
(ดูหน้า dataset บน Kaggle ว่าเจ้าของบอกที่มาไว้ว่าอย่างไร)

In [ ]:
labels = {
    "Age": "อายุ",
    "Avg_Daily_Usage_Hours": "เวลาใช้โซเชียล",
    "Daily_Unlocks": "ปลดล็อกมือถือ",
    "Study_Hours": "เวลาอ่านหนังสือ",
    "Physical_Activity_Hours": "กิจกรรมทางกาย",
    "Sleep_Hours_Per_Night": "เวลานอน",
    "Mental_Health_Score": "คะแนนสุขภาพจิต",
}

corr = df[list(labels)].corr().rename(index=labels, columns=labels)
corr = corr.iloc[1:, :-1]                                  # ตัดเส้นทแยง (1.00 เสมอ) ออก
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)        # ครึ่งบนเป็นค่าซ้ำ ไม่ต้องโชว์

fig, ax = plt.subplots(figsize=(7.5, 5.5))
sns.heatmap(corr, mask=mask, cmap="coolwarm", vmin=-1, vmax=1, center=0,
            annot=True, fmt=".2f", annot_kws={"fontsize": "small"},
            linewidths=2, linecolor="white", square=True,
            cbar_kws={"shrink": 0.75, "label": "สหสัมพันธ์เพียร์สัน (r)"}, ax=ax)

ax.set_title("ความสัมพันธ์ระหว่างตัวแปรเชิงตัวเลข")
ax.tick_params(length=0)
plt.setp(ax.get_xticklabels(), rotation=30, ha="right")
plt.show()

## 5. คู่ที่น่าสนใจที่สุด: เวลาใช้งาน × คะแนนสุขภาพจิต

จุดเยอะถึง 5,000 จุด จะทับกันจนอ่านไม่ออก เลยต้องลด `alpha` ลง
แล้วลากเส้นค่าเฉลี่ยรายช่วงทับไว้ — เส้นนี้แหละคือสิ่งที่อยากให้คนอ่านเห็น

In [ ]:
fig, ax = plt.subplots(figsize=(7.2, 4.6))

ax.scatter(df["Avg_Daily_Usage_Hours"], df["Mental_Health_Score"],
           s=14, color=BLUE, alpha=0.18, edgecolor="none", label="นักศึกษา 1 คน")

# แบ่งชั่วโมงเป็นช่วงละ 1 ชม. แล้วหาค่าเฉลี่ยของแต่ละช่วง
bins = np.arange(0, df["Avg_Daily_Usage_Hours"].max() + 1, 1.0)
mid = bins[:-1] + 0.5
g = df.groupby(pd.cut(df["Avg_Daily_Usage_Hours"], bins), observed=False)["Mental_Health_Score"]
stat = g.agg(["mean", "count"])
solid = (stat["count"] >= 20).to_numpy()   # ช่วงที่คนน้อยเกินไป ไม่ลากเส้น

ax.plot(mid[solid], stat.loc[solid, "mean"], color=ORANGE, marker="o",
        markeredgecolor="white", markeredgewidth=1.5, label="ค่าเฉลี่ยของแต่ละช่วง")

ax.set_title("ยิ่งใช้โซเชียลนาน คะแนนสุขภาพจิตยิ่งต่ำ")
ax.set_xlabel("ชั่วโมงต่อวัน")
ax.set_ylabel("คะแนนสุขภาพจิต (ยิ่งสูง = ยิ่งดี)")
ax.xaxis.set_major_locator(mticker.MultipleLocator(2))
ax.legend(loc="upper right", frameon=False)

r = df["Avg_Daily_Usage_Hours"].corr(df["Mental_Health_Score"])
print(f"r = {r:.2f}")
plt.show()

## 6. ที่ว่างสำหรับลองเอง

คำถามที่กราฟน่าจะตอบได้ เลือกสักข้อแล้ววาดดู:

1. กลุ่ม `Stress_Level` ทั้ง 4 ระดับ ใช้โซเชียลต่างกันแค่ไหน (`boxplot` เทียบการกระจาย)
2. แพลตฟอร์มไหนคะแนนสุขภาพจิตเฉลี่ยต่ำสุด — แต่ต้องระวัง แพลตฟอร์มที่มีคนตอบแค่ 36 คน
   (WeChat, KakaoTalk) ค่าเฉลี่ยจะแกว่งมาก ควรกรองเอาเฉพาะที่ n เยอะพอ
3. เวลานอนเป็นตัวกลางหรือเปล่า — คนที่ใช้โซเชียลเท่ากันแต่นอนต่างกัน คะแนนสุขภาพจิตต่างกันไหม
4. `Academic_Level` แต่ละระดับต่างกันไหม

ท่าที่ใช้บ่อย:

```python
order = ["Low", "Medium", "High", "Very High"]          # บังคับลำดับกลุ่ม ไม่งั้นเรียงมั่ว
sns.boxplot(df, x="Stress_Level", y="Avg_Daily_Usage_Hours", order=order)
sns.barplot(df, x=..., y=..., errorbar="ci")            # ค่าเฉลี่ย + ช่วงความเชื่อมั่น
df.groupby("Most_Used_Platform")["Mental_Health_Score"].agg(["mean", "count"])
pd.cut(df["Sleep_Hours_Per_Night"], [0, 6, 8, 24])      # แบ่งตัวเลขเป็นกลุ่ม
```

**ข้อควรระวังตอนเขียนสรุป:** ทั้งหมดนี้คือ *ความสัมพันธ์* ไม่ใช่ *สาเหตุ*
ข้อมูลนี้เก็บครั้งเดียว (cross-sectional) บอกไม่ได้ว่าใช้โซเชียลเยอะทำให้สุขภาพจิตแย่
หรือคนที่สุขภาพจิตแย่อยู่แล้วหันไปเล่นโซเชียลมากกว่า

In [ ]:
# ตัวช่วยเริ่มต้นข้อ 2 — เอาเฉพาะแพลตฟอร์มที่มีคนตอบตั้งแต่ 100 คนขึ้นไป
stats = (df.groupby("Most_Used_Platform")["Mental_Health_Score"]
           .agg(["mean", "count"])
           .query("count >= 100")
           .sort_values("mean"))
stats.round(2)

# ต่อเอง: เอา stats ไปวาดเป็นแท่งแนวนอน (ax.barh) แล้วติดตัวเลขไว้ที่ปลายแท่ง

## 7. เอารูปไปใส่ในเล่ม

เซฟลง `proposal/figures/` ได้เลย แล้วอ้างใน LaTeX — `main.tex` ตั้ง `\graphicspath{{figures/}}` ไว้แล้ว
จึงใส่แค่ชื่อไฟล์:

```latex
\begin{figure}[H]
  \centering
  \includegraphics[width=0.8\textwidth]{usage-vs-mental-health.png}
  \caption{ความสัมพันธ์ระหว่างเวลาใช้โซเชียลมีเดียกับคะแนนสุขภาพจิต}
\end{figure}
```

**ข้อควรระวังของ notebook:** ถ้ารันเซลล์สลับลำดับ ตัวแปรจะค้างจากการรันครั้งก่อนได้
ก่อนเซฟรูปจริงที่จะส่งอาจารย์ ให้กด Kernel → Restart Kernel and Run All Cells
แล้วดูว่าผลยังเหมือนเดิมไหม

In [ ]:
FIGURES = Path.cwd().parents[1] / "figures"   # proposal/figures/
FIGURES.mkdir(exist_ok=True)

fig.savefig(FIGURES / "usage-vs-mental-health.png")   # fig = รูปล่าสุดที่วาดไว้ข้างบน
print("เซฟแล้ว:", FIGURES / "usage-vs-mental-health.png")